In [ ]:
import sys, os, yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm đường dẫn src vào hệ thống
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.supervised import prepare_classification_data, train_classifiers
from src.models.handler import save_model_artifact
from src.visualization.plots import plot_confusion_matrix
from sklearn.metrics import classification_report, f1_score

# Load Config
with open('../configs/params.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Đọc dữ liệu đã tiền xử lý (đã có Month, Season, Precip Type='none'...)
df = pd.read_csv(f"../{config['processed_data_path']}")

In [8]:
# 2. Split Data


(X_train, X_test, y_train, y_test), le_target = prepare_classification_data(df, config)

In [9]:
# Lấy số lượng nhãn thực tế sau khi lọc
num_labels = len(le_target.classes_)
# 3. Train Models
rf_model = train_classifiers(X_train, y_train, config, num_classes=num_labels)


d:\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:45:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [10]:
# 4. Đánh giá F1-macro 
y_pred_rf = rf_model.predict(X_test)
print("--- KẾT QUẢ RANDOM FOREST ---")

print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))

AttributeError: 'tuple' object has no attribute 'predict'

In [ ]:
# 5. Phân tích lỗi giao mùa/cực trị (Yêu cầu bài toán)
# Lọc những dòng dự báo sai
results = X_test.copy()
results['Actual'] = le_target.inverse_transform(y_test)
results['Predicted'] = le_target.inverse_transform(y_pred_rf)
errors = results[results['Actual'] != results['Predicted']]

print(f"Số lượng lỗi: {len(errors)}")
# Xem lỗi tập trung vào tháng nào (Giao mùa: tháng 3, 4 hoặc 10, 11)
print("Thống kê lỗi theo tháng:")
print(errors['Month'].value_counts().sort_index())



Số lượng lỗi: 82
Thống kê lỗi theo tháng:
Month
4     16
5     16
10    48
11     2
Name: count, dtype: int64


In [ ]:
# 6. Lưu mô hình
save_model_artifact(rf_model, "rf_weather_model", config)

## XGBoost

In [11]:




# 2. Huấn luyện cả 2 mô hình
rf_model, xgb_model = train_classifiers(X_train, y_train, config, num_classes=num_labels)

# 3. Đánh giá XGBoost (So sánh với RF)
y_pred_xgb = xgb_model.predict(X_test)

print("--- KẾT QUẢ XGBOOST (DỰ BÁO SUMMARY) ---")
from sklearn.metrics import classification_report, f1_score
print(classification_report(y_test, y_pred_xgb, target_names=le_target.classes_))

# Tính F1-macro cụ thể để so sánh (Yêu cầu bài toán)
f1_rf = f1_score(y_test, rf_model.predict(X_test), average='macro')
f1_xgb = f1_score(y_test, y_pred_xgb, average='macro')

print(f"F1-Macro Score - Random Forest: {f1_rf:.4f}")
print(f"F1-Macro Score - XGBoost: {f1_xgb:.4f}")

# 4. Lưu cả 2 mô hình vào outputs/models/
from src.models.handler import save_model_artifact
save_model_artifact(rf_model, "rf_summary_model", config)
save_model_artifact(xgb_model, "xgb_summary_model", config)

d:\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:46:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- KẾT QUẢ XGBOOST (DỰ BÁO SUMMARY) ---
                          precision    recall  f1-score   support

Breezy and Mostly Cloudy       0.41      0.47      0.44        92
     Breezy and Overcast       0.60      0.61      0.60       112
Breezy and Partly Cloudy       0.60      0.58      0.59        84
                   Clear       0.56      0.26      0.36      2164
                   Foggy       0.99      0.99      0.99      1406
           Mostly Cloudy       0.48      0.45      0.47      5732
                Overcast       0.56      0.50      0.53      3269
           Partly Cloudy       0.53      0.69      0.60      6320

                accuracy                           0.56     19179
               macro avg       0.59      0.57      0.57     19179
            weighted avg       0.56      0.56      0.55     19179

F1-Macro Score - Random Forest: 0.5653
F1-Macro Score - XGBoost: 0.5717
📦 Đã lưu mô hình: ../outputs/models/rf_summary_model.pkl
📦 Đã lưu mô hình: ../outputs/models

In [ ]:
# 6. Lưu mô hình

save_model_artifact(xgb_model, "xgb_weather_model", config)

📦 Đã lưu mô hình: ../outputs/models/rf_weather_model.pkl
📦 Đã lưu mô hình: ../outputs/models/xgb_weather_model.pkl
